# 01 — ETL and Data Warehouse

Loads the raw Olist CSVs, cleans them, and builds a star schema in **DuckDB** (swapped from PostgreSQL for this Colab environment -- DuckDB is a single embedded file that persists directly to Drive, with no service to start and no session-reset issues).

Run this notebook first. It writes `warehouse.duckdb` and cleaned parquet files that every later notebook reads.

In [1]:

from google.colab import drive
drive.mount('/content/drive')

!pip install -q duckdb pandas pyarrow scikit-learn xgboost lightgbm mlxtend shap prophet google-genai

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROCESSED_DIR / "features"
MODELS_DIR = PROCESSED_DIR / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
WAREHOUSE_PATH = PROJECT_ROOT / "warehouse.duckdb"

for d in [PROCESSED_DIR, FEATURES_DIR, MODELS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


Mounted at /content/drive
Project root: /content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence


## Extract

Raw Olist CSVs must already be in `DATA_DIR` (download from Kaggle and upload to Drive if not present).

In [2]:

import pandas as pd

RAW_FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

raw = {}
for key, filename in RAW_FILES.items():
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing {path} -- upload the Olist CSVs to DATA_DIR first.")
    raw[key] = pd.read_csv(path)
    print(f"{key:22s} {len(raw[key]):>7,} rows")


orders                  99,441 rows
order_items            112,650 rows
order_payments         103,886 rows
order_reviews           99,224 rows
customers               99,441 rows
products                32,951 rows
sellers                  3,095 rows
category_translation        71 rows


## Transform

Dedup, dtype fixes, missing-value handling, category translation merge. `recency`/derived fields are computed later in feature engineering, not here.

In [3]:

VALID_ORDER_STATUSES = ["delivered", "shipped", "canceled", "invoiced", "processing", "unavailable", "approved"]
DATE_COLS = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]

def clean_orders(orders):
    orders = orders.copy()
    for col in DATE_COLS:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")
    orders = orders.drop_duplicates(subset="order_id")
    orders = orders[orders["order_status"].isin(VALID_ORDER_STATUSES)]
    orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
    orders["delivery_delay_days"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days
    return orders

def clean_order_items(order_items):
    order_items = order_items.dropna(subset=["order_id", "product_id", "seller_id", "price"]).copy()
    order_items = order_items.drop_duplicates(subset=["order_id", "order_item_id"])
    order_items["price"] = order_items["price"].astype(float)
    order_items["freight_value"] = order_items["freight_value"].astype(float)
    return order_items

def clean_payments(payments):
    return (payments.drop_duplicates().groupby("order_id").agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_type=("payment_type", lambda x: x.mode().iat[0] if not x.mode().empty else "unknown"),
    ).reset_index())

def clean_reviews(reviews):
    reviews = reviews.drop_duplicates(subset="order_id", keep="last")[["order_id", "review_score"]].copy()
    reviews["review_score"] = reviews["review_score"].fillna(reviews["review_score"].median())
    return reviews

def clean_products(products, category_translation):
    products = products.merge(category_translation, on="product_category_name", how="left").copy()
    products["product_category_name"] = products["product_category_name"].fillna("unknown")
    products["product_category_name_english"] = products["product_category_name_english"].fillna("unknown")
    for col in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
        products[col] = products[col].fillna(products[col].median())
    return products

def clean_customers(customers):
    return customers.drop_duplicates(subset="customer_id").copy()

def clean_sellers(sellers):
    return sellers.drop_duplicates(subset="seller_id").copy()

orders = clean_orders(raw["orders"])
order_items = clean_order_items(raw["order_items"])
payments = clean_payments(raw["order_payments"])
reviews = clean_reviews(raw["order_reviews"])
products = clean_products(raw["products"], raw["category_translation"])
customers = clean_customers(raw["customers"])
sellers = clean_sellers(raw["sellers"])

print("Cleaning complete:")
for name, df in [("orders", orders), ("order_items", order_items), ("payments", payments),
                  ("reviews", reviews), ("products", products), ("customers", customers), ("sellers", sellers)]:
    print(f"  {name:12s} {len(df):>7,} rows")


Cleaning complete:
  orders        99,436 rows
  order_items  112,650 rows
  payments      99,440 rows
  reviews       98,673 rows
  products      32,951 rows
  customers     99,441 rows
  sellers        3,095 rows


## Load: build the star schema in DuckDB

`Fact_OrderItem` (order-item grain) + 6 dimensions: Customer, Product, Seller, Payment, Date, Review. DuckDB can query pandas DataFrames directly (`register`), so no intermediate CSV export is needed.

In [4]:

import duckdb

con = duckdb.connect(str(WAREHOUSE_PATH))

con.register("orders_df", orders)
con.register("order_items_df", order_items)
con.register("payments_df", payments)
con.register("reviews_df", reviews)
con.register("products_df", products)
con.register("customers_df", customers)
con.register("sellers_df", sellers)

con.execute("""
    CREATE OR REPLACE TABLE dim_customer AS
    SELECT ROW_NUMBER() OVER () AS customer_key, customer_id, customer_unique_id,
           customer_city, customer_state, customer_zip_code_prefix AS customer_zip_prefix
    FROM customers_df
""")
con.execute("""
    CREATE OR REPLACE TABLE dim_product AS
    SELECT ROW_NUMBER() OVER () AS product_key, product_id,
           product_category_name AS category_name,
           product_category_name_english AS category_name_english,
           product_weight_g AS weight_g, product_length_cm AS length_cm,
           product_height_cm AS height_cm, product_width_cm AS width_cm
    FROM products_df
""")
con.execute("""
    CREATE OR REPLACE TABLE dim_seller AS
    SELECT ROW_NUMBER() OVER () AS seller_key, seller_id, seller_city, seller_state,
           seller_zip_code_prefix AS seller_zip_prefix
    FROM sellers_df
""")
con.execute("""
    CREATE OR REPLACE TABLE dim_payment AS
    SELECT ROW_NUMBER() OVER () AS payment_key, payment_type
    FROM (SELECT DISTINCT payment_type FROM payments_df)
""")
con.execute("""
    CREATE OR REPLACE TABLE dim_review AS
    SELECT ROW_NUMBER() OVER () AS review_key, order_id, review_score
    FROM reviews_df
""")
con.execute("""
    CREATE OR REPLACE TABLE dim_date AS
    SELECT ROW_NUMBER() OVER () AS date_key_unused,
           CAST(strftime(d, '%Y%m%d') AS INTEGER) AS date_key,
           d AS full_date, EXTRACT(day FROM d) AS day, EXTRACT(month FROM d) AS month,
           EXTRACT(quarter FROM d) AS quarter, EXTRACT(year FROM d) AS year,
           EXTRACT(dow FROM d) AS day_of_week,
           EXTRACT(dow FROM d) IN (0, 6) AS is_weekend
    FROM (SELECT UNNEST(generate_series(
            (SELECT MIN(order_purchase_timestamp)::DATE FROM orders_df),
            (SELECT MAX(order_purchase_timestamp)::DATE FROM orders_df),
            INTERVAL 1 DAY)) AS d)
""")

con.execute("""
    CREATE OR REPLACE TABLE fact_order_item AS
    SELECT
        oi.order_id, oi.order_item_id,
        c.customer_key, p.product_key, s.seller_key, pay.payment_key,
        CAST(strftime(o.order_purchase_timestamp, '%Y%m%d') AS INTEGER) AS order_date_key,
        r.review_key,
        oi.price, oi.freight_value,
        COALESCE(pm.payment_value, 0.0) AS payment_value,
        COALESCE(pm.payment_installments, 1) AS payment_installments,
        o.delivery_days, o.delivery_delay_days
    FROM order_items_df oi
    JOIN orders_df o ON oi.order_id = o.order_id
    LEFT JOIN dim_customer c ON o.customer_id = c.customer_id
    LEFT JOIN dim_product p ON oi.product_id = p.product_id
    LEFT JOIN dim_seller s ON oi.seller_id = s.seller_id
    LEFT JOIN payments_df pm ON oi.order_id = pm.order_id
    LEFT JOIN dim_payment pay ON pm.payment_type = pay.payment_type
    LEFT JOIN dim_review r ON oi.order_id = r.order_id
""")

for t in ["dim_customer", "dim_product", "dim_seller", "dim_payment", "dim_review", "dim_date", "fact_order_item"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:18s} {n:>8,} rows")

con.close()
print("\nWarehouse saved to", WAREHOUSE_PATH)


dim_customer         99,441 rows
dim_product          32,951 rows
dim_seller            3,095 rows
dim_payment               5 rows
dim_review           98,673 rows
dim_date                774 rows
fact_order_item     112,650 rows

Warehouse saved to /content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence/warehouse.duckdb


## OLAP sample queries

Roll-up, drill-down, slice, dice, pivot -- same queries as the original `olap_queries.sql`.

In [5]:

con = duckdb.connect(str(WAREHOUSE_PATH))

print("ROLL-UP: revenue by year -> quarter")
display(con.execute("""
    SELECT d.year, d.quarter, ROUND(SUM(f.price), 2) AS revenue
    FROM fact_order_item f JOIN dim_date d ON f.order_date_key = d.date_key
    GROUP BY ROLLUP (d.year, d.quarter) ORDER BY d.year, d.quarter
""").df())

print("SLICE: category revenue for Q4")
display(con.execute("""
    SELECT p.category_name_english, ROUND(SUM(f.price), 2) AS revenue
    FROM fact_order_item f
    JOIN dim_product p ON f.product_key = p.product_key
    JOIN dim_date d ON f.order_date_key = d.date_key
    WHERE d.quarter = 4
    GROUP BY p.category_name_english ORDER BY revenue DESC LIMIT 10
""").df())

con.close()


ROLL-UP: revenue by year -> quarter


,year,quarter,revenue
0,2016,3,267.36
1,2016,4,49518.56
2,2016,<NA>,49785.92
3,2017,1,741960.19
4,2017,2,1299036.97
5,2017,3,1696404.85
6,2017,4,2418404.97
7,2017,<NA>,6155806.98
8,2018,1,2777422.51
9,2018,2,2858289.74


SLICE: category revenue for Q4


,category_name_english,revenue
0,watches_gifts,238771.96
1,health_beauty,186853.29
2,bed_bath_table,186595.38
3,sports_leisure,177751.80
4,toys,160613.13
5,computers_accessories,155958.06
6,cool_stuff,141327.79
7,furniture_decor,131271.12
8,garden_tools,101000.49
9,auto,97581.55


## Save cleaned tables to parquet

Used directly by Notebook 2 (feature engineering).

In [6]:

customers.to_parquet(PROCESSED_DIR / "customers.parquet", index=False)
orders.to_parquet(PROCESSED_DIR / "orders.parquet", index=False)
order_items.to_parquet(PROCESSED_DIR / "order_items.parquet", index=False)
payments.to_parquet(PROCESSED_DIR / "payments.parquet", index=False)
reviews.to_parquet(PROCESSED_DIR / "reviews.parquet", index=False)
products.to_parquet(PROCESSED_DIR / "products.parquet", index=False)
sellers.to_parquet(PROCESSED_DIR / "sellers.parquet", index=False)
print("Saved cleaned parquet files to", PROCESSED_DIR)


Saved cleaned parquet files to /content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence/data/processed
